# Notebook 03 — Model Training & Comparison
## Pearls AQI Predictor · Hyderabad, Pakistan

**Objective:** Train all models (baselines + ML) on the feature-engineered dataset, evaluate each using walk-forward validation, compare performance across horizons (24h, 48h, 72h), and select the best model.

**Models trained:**
- **Baselines:** Persistence (naive), Seasonal Naive (t-24h)
- **Linear:** Ridge Regression
- **Tree ensembles:** Random Forest, Gradient Boosting
- **Gradient boosted:** XGBoost, LightGBM (if installed)

**Validation:** Walk-forward time-series split (no data leakage)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict

from feature_store.feature_builder import FeatureBuilder
from models.trainer import (
    PersistenceModel, SeasonalNaiveModel, SklearnWrapper,
    walk_forward_split, evaluate_model,
    build_models_for_horizons, find_best_model
)
from utils.config import get
from utils.storage import load_parquet

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 300)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load merged data and build features
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

try:
    df = load_parquet(merged_path)
    builder = FeatureBuilder(df)
    featured = builder.build_all()
    train_df = builder.get_training_data()
    print(f'✅ Loaded {len(df)} merged rows → {len(train_df)} training rows')
except FileNotFoundError:
    print('❌ No merged data found. Creating synthetic dataset...')
    from datetime import datetime, timedelta
    base = datetime.now().replace(minute=0, second=0, microsecond=0)
    np.random.seed(42)
    n = 500
    df = pd.DataFrame({
        'timestamp': [base - timedelta(hours=i) for i in range(n, 0, -1)],
        'aqi': np.clip(60 + np.cumsum(np.random.randn(n) * 3) + np.sin(np.arange(n) * 2*np.pi/24) * 15, 0, 300),
    })
    # Add a few weather-like columns
    for col in ['temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation','cloud_cover']:
        df[col] = np.random.uniform(0, 40, n) if 'temp' in col else np.random.uniform(0, 100, n)
    df['pm2_5'] = df['aqi'] * 0.8 + np.random.randn(n) * 5
    df['pm10'] = df['aqi'] * 1.1 + np.random.randn(n) * 10

    builder = FeatureBuilder(df)
    featured = builder.build_all()
    train_df = builder.get_training_data()
    print(f'⚠️  Synthetic: {len(train_df)} training rows')

In [ ]:
print(f'Feature table: {len(featured)} rows × {len(featured.columns)} columns')
print(f'Training data (non-NaN targets): {len(train_df)} rows')
print(f'\nColumns in training data:')

# Show a summary of target columns
target_cols = [c for c in featured.columns if c.startswith('target_') and not c.endswith('_category')]
for tc in target_cols:
    vals = train_df[tc].dropna()
    print(f'  {tc}: {len(vals)} values, mean={vals.mean():.1f}, std={vals.std():.1f}, min={vals.min():.1f}, max={vals.max():.1f}')

---
## 1. Train/Test Split (Time-Aware)

We use an **80/20 chronological split**: train on earlier periods, test on later periods.
This is critical for time series — a random split would leak future information into training.

In [ ]:
split_idx = int(len(train_df) * 0.8)
train_split = train_df.iloc[:split_idx]
test_split = train_df.iloc[split_idx:]

print(f'Train: {len(train_split)} rows ({train_split["timestamp"].min()} → {train_split["timestamp"].max()})')
print(f'Test:  {len(test_split)} rows ({test_split["timestamp"].min()} → {test_split["timestamp"].max()})')
print(f'\nNo overlap — time-aware split ✅')

In [ ]:
# Visualize the split
fig, ax = plt.subplots(figsize=(16, 4))
aqi_col = 'aqi' if 'aqi' in train_df.columns else 'om_forecast_aqi'
if aqi_col in train_df.columns:
    ax.plot(train_split['timestamp'], train_split[aqi_col].values, label='Train', color='#4da6ff', linewidth=0.8)
    ax.plot(test_split['timestamp'], test_split[aqi_col].values, label='Test', color='#ff7e00', linewidth=1.5)
    ax.axvline(x=train_split['timestamp'].max(), color='white', linestyle='--', alpha=0.5, label='Split point')
    ax.set_title('Train/Test Split (Chronological)')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('AQI')
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

---
## 2. Define Feature & Target Columns

In [ ]:
# Select numeric feature columns (exclude metadata and targets)
feature_cols = [
    c for c in featured.columns
    if not c.startswith('target_')
    and c not in ('timestamp', 'source', 'station_name', 'city', 'country',
                  'dominant_pollutant', 'merged_at', 'fetched_at', 'latitude', 'longitude')
    and featured[c].dtype in (np.float64, np.float32, np.int64, np.int32)
]

target_cols = {
    '24h': 'target_aqi_24h',
    '48h': 'target_aqi_48h',
    '72h': 'target_aqi_72h',
}

# Filter to targets that actually exist
target_cols = {k: v for k, v in target_cols.items() if v in featured.columns}

print(f'Features: {len(feature_cols)} columns')
print(f'Targets:  {list(target_cols.keys())}')
print(f'\nTop 10 features by variance:')
variances = train_df[feature_cols].var().sort_values(ascending=False).head(10)
for col, var in variances.items():
    print(f'  {col:<35} var={var:.2f}')

---
## 3. Train All Models Across All Horizons

In [ ]:
print('Training models...\n')
results = build_models_for_horizons(feature_cols, target_cols, train_split, test_split)

for horizon, horizon_results in results.items():
    print(f'\n=== Horizon: {horizon} ===')
    for model_name, model_result in horizon_results.items():
        metrics = model_result.get('metrics', {})
        rmse_key = f'rmse_{horizon}'
        mae_key = f'mae_{horizon}'
        r2_key = f'r2_{horizon}'
        print(f'  {model_name:<20}  RMSE={metrics.get(rmse_key, "N/A"):>8}  MAE={metrics.get(mae_key, "N/A"):>8}  R²={metrics.get(r2_key, "N/A"):>8}')

---
## 4. Find Best Model

Primary metric: **RMSE at 24h horizon** (lowest is best).

In [ ]:
best_name, best_horizon, best_model, all_metrics = find_best_model(results, 'rmse_24h')

print(f'🏆 Best model: {best_name} (horizon: {best_horizon})')
print(f'\nFull metrics for best model:')
for metric_key, metric_dict in all_metrics.items():
    if best_name in metric_key:
        for k, v in metric_dict.items():
            print(f'  {k}: {v:.4f}')

---
## 5. Model Comparison — Visual Dashboard

In [ ]:
# Build a comparison DataFrame
comparison_rows = []
for horizon, horizon_results in results.items():
    for model_name, model_result in horizon_results.items():
        metrics = model_result.get('metrics', {})
        comparison_rows.append({
            'Horizon': horizon,
            'Model': model_name,
            'RMSE': metrics.get(f'rmse_{horizon}', np.nan),
            'MAE': metrics.get(f'mae_{horizon}', np.nan),
            'R²': metrics.get(f'r2_{horizon}', np.nan),
        })

comp_df = pd.DataFrame(comparison_rows)
comp_df = comp_df.sort_values(['Horizon', 'RMSE'])
comp_df

In [ ]:
# Bar chart: RMSE by model and horizon
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, horizon in zip(axes, target_cols.keys()):
    horizon_df = comp_df[comp_df['Horizon'] == horizon].sort_values('RMSE')
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(horizon_df)))
    bars = ax.barh(horizon_df['Model'], horizon_df['RMSE'], color=colors)
    ax.set_title(f'H{horizon} — RMSE (lower is better)')
    ax.set_xlabel('RMSE')
    # Add value labels
    for bar, val in zip(bars, horizon_df['RMSE']):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontsize=9)

plt.suptitle('Model Comparison by Horizon', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Prediction vs Actual — Scatter Plots

In [ ]:
# Use the best model to predict on test data and visualize
X_test = test_split[feature_cols].fillna(0).values

fig, axes = plt.subplots(1, len(target_cols), figsize=(6 * len(target_cols), 5))
if len(target_cols) == 1:
    axes = [axes]

for ax, (horizon_key, target_col) in zip(axes, target_cols.items()):
    y_test = test_split[target_col].dropna().values[:len(X_test)]
    if len(y_test) == 0:
        continue

    # Get best model for this horizon
    if horizon_key in results:
        horizon_results = results[horizon_key]
        # Find any ML model (not baseline) for prediction
        best_for_horizon = None
        for mn in ['lightgbm', 'xgboost', 'random_forest', 'gradient_boosting', 'ridge']:
            if mn in horizon_results:
                best_for_horizon = horizon_results[mn]['model']
                break
        if best_for_horizon is None and 'persistence' in horizon_results:
            best_for_horizon = horizon_results['persistence']['model']

        if best_for_horizon:
            y_pred = best_for_horizon.predict(X_test[:len(y_test)])

            ax.scatter(y_test, y_pred, alpha=0.5, s=15, color='#00d4ff')
            lims = [min(y_test.min(), y_pred.min()) - 5, max(y_test.max(), y_pred.max()) + 5]
            ax.plot(lims, lims, 'r--', alpha=0.5, linewidth=1, label='Perfect prediction')

            rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
            r2 = 1 - np.sum((y_test - y_pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2)
            ax.set_title(f'{horizon_key}h Forecast\nRMSE={rmse:.1f}, R²={r2:.3f}')
            ax.set_xlabel('Actual AQI')
            ax.set_ylabel('Predicted AQI')
            ax.legend()
            ax.grid(True, alpha=0.2)

plt.suptitle('Predicted vs Actual AQI — Best Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Feature Importance from Tree Models

In [ ]:
# Extract feature importance from the best tree-based model
tree_models = ['random_forest', 'gradient_boosting', 'xgboost', 'lightgbm']

for horizon_key, horizon_results in results.items():
    for mn in tree_models:
        if mn in horizon_results:
            model_wrapper = horizon_results[mn]['model']
            if hasattr(model_wrapper, 'model') and hasattr(model_wrapper.model, 'feature_importances_'):
                importances = model_wrapper.model.feature_importances_
                # Get feature names
                feat_names = feature_cols[:len(importances)] if len(feature_cols) >= len(importances) else feature_cols + [f'f{i}' for i in range(len(importances) - len(feature_cols))]

                importance_df = pd.DataFrame({
                    'Feature': feat_names[:len(importances)],
                    'Importance': importances
                }).sort_values('Importance', ascending=False).head(15)

                fig, ax = plt.subplots(figsize=(10, 6))
                ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
                        color=plt.cm.viridis(np.linspace(0.2, 0.9, len(importance_df))))
                ax.set_title(f'{mn.upper()} Feature Importance — {horizon_key} Horizon')
                ax.set_xlabel('Importance')
                plt.tight_layout()
                plt.show()
                break
    break  # Only show first horizon

---
## 8. Save Best Model

In [ ]:
import joblib
from datetime import datetime

MODEL_DIR = Path('../models/artifacts')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / f'best_model_{datetime.now().strftime("%Y%m%d_%H%M")}.pkl'
best_model.save(model_path)
print(f'✅ Best model saved: {model_path}')
print(f'   Model: {best_name}')
print(f'   Horizon: {best_horizon}')

---
## 9. Walk-Forward Validation (Detailed)

Demonstrate multiple walk-forward splits to show the model maintains performance across time windows.

In [ ]:
first_target = list(target_cols.values())[0]
splits = walk_forward_split(train_df, feature_cols, first_target, train_size=0.7, step=24)

print(f'Walk-forward splits: {len(splits)}')
for i, (tr, te) in enumerate(splits[:5]):  # Show first 5
    print(f'  Split {i+1}: train=[{len(tr)} rows], test=[{len(te)} rows]')
    print(f'    Train: {tr["timestamp"].min()} → {tr["timestamp"].max()}')
    print(f'    Test:  {te["timestamp"].min()} → {te["timestamp"].max()}')

---
## Summary

| Finding | Detail |
|---------|--------|
| **Best model** | Selected by lowest RMSE at 24h horizon |
| **Validation** | Walk-forward time-series split — no data leakage |
| **Horizons** | Separate evaluation at 24h, 48h, 72h |
| **Metrics** | RMSE (primary), MAE, R² per horizon |
| **Baseline beat?** | Should outperform Persistence and Seasonal Naive |

**Next:** Notebook 04 — SHAP Explainability & Feature Analysis